# ML-ESS — Backend Walkthrough

A guided tour of the production backend located in `api/`.  
Each agent stage is exercised individually so its output can be inspected
before being passed to the next stage, followed by a full end-to-end
`run_pipeline()` demonstration.

## Pipeline architecture

```
Research Question
      │
      ▼
 ┌────────────┐   search_queries, sources, evidence
 │ SearchAgent│ ─────────────────────────────────────────────┐
 └────────────┘                                              │
                                                             ▼
                                                  ┌──────────────────┐
                                                  │ SynthesisAgent   │
                                                  │ themes +         │
                                                  │ contradictions   │
                                                  └────────┬─────────┘
                                                           │
                                                           ▼
                                                  ┌──────────────────┐
                                                  │  ReportAgent     │
                                                  │  final_report    │
                                                  └────────┬─────────┘
                                                           │
                                                           ▼
                                                  ┌──────────────────┐
                                                  │   Evaluator      │
                                                  │ EvaluationScores │
                                                  └────────┬─────────┘
                                                           │
                                                           ▼
                                                      SharedState
```

| Variable | Default | Description |
|---|---|---|
| `LLM_PROVIDER` | `auto` | `groq` / `huggingface` / `auto` |
| `GROQ_API_KEY` | — | Required when provider = groq |
| `TAVILY_API_KEY` | — | Enables Tavily search; DDG used as fallback |
| `MAX_SEARCH_QUERIES` | `3` | Queries generated per question |
| `MAX_RESULTS_PER_QUERY` | `5` | Web results fetched per query |

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# Install all packages required by the api/ backend.
%pip install -qU groq tavily-python duckduckgo-search httpx beautifulsoup4 \
    pydantic python-dotenv huggingface_hub markdown matplotlib pandas

In [ ]:
"""Environment setup — adds api/ to the module search path and configures
logging so agent log lines appear inline in the notebook output.
"""

from __future__ import annotations

import logging
import sys
from pathlib import Path

# ── Module path ───────────────────────────────────────────────────────────────
# Resolve api/ relative to this notebook so `from app.xxx import yyy` works
# without installing the package.
_api_dir = Path("../../api").resolve()
if str(_api_dir) not in sys.path:
    sys.path.insert(0, str(_api_dir))

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)-8s] %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger(__name__)
logger.info("api/ resolved to: %s", _api_dir)

---
## Data Models

`SharedState` is the single object every agent reads from and writes to.
Its sub-models accumulate as the pipeline progresses:

| Stage | Populates |
|---|---|
| SearchAgent | `search_queries`, `sources`, `evidence` |
| SynthesisAgent | `themes`, `contradictions` |
| ReportAgent | `report_outline`, `final_report` |
| Evaluator | `evaluation` |

In [ ]:
# ── Data Models ───────────────────────────────────────────────────────────────
from app.models.state import (
    Confidence,
    Contradiction,
    Evidence,
    EvaluationScores,
    SharedState,
    Source,
    Theme,
)
from app.models.api import JobStatus

# Instantiate an empty state to inspect its schema.
demo = SharedState(research_question="What is quantum entanglement?")
print(demo.summary())

---
## LLM Client

`get_client()` selects a provider based on the `LLM_PROVIDER` env var and
returns an object satisfying the `LLMClient` protocol (`completions(system, user) -> str`).

The `chat()` helper wraps it with retries and automatic provider fallback.

In [ ]:
# ── LLM Client ────────────────────────────────────────────────────────────────
from app.core.llm import chat, get_client

client = get_client()
logger.info("Active provider: %s", client.provider)

# Smoke-test: a single round-trip to confirm the key and model are working.
reply = chat(client, "You are a helpful assistant.", "Say hello in one sentence.")
print(reply)

---
## Stage-by-Stage Walkthrough

Each stage accepts a `SharedState`, mutates it, and returns it.  
An optional `on_event` callback receives structured progress events.

All four stages share the same `state` object defined here.

In [ ]:
# ── Research question ─────────────────────────────────────────────────────────
RESEARCH_QUESTION = (
    "What are the main trade-offs between CNNs and "
    "Vision Transformers for medical imaging?"
)

state = SharedState(research_question=RESEARCH_QUESTION)

# Collect all events emitted by agents for later inspection.
event_log: list[tuple[str, dict]] = []

def _on_event(name: str, data: dict | None = None) -> None:
    """Append every agent event to event_log and log it."""
    event_log.append((name, data or {}))
    logger.info("  event: %s", name)

### Stage 1 — SearchAgent

Generates search queries with the LLM, fetches web pages (Tavily → DuckDuckGo
fallback), scrapes full-page text with BeautifulSoup, and extracts structured
evidence fragments from each source.

In [ ]:
# ── Stage 1: SearchAgent ──────────────────────────────────────────────────────
from app.agents import search

state = search.run(state, on_event=_on_event)

logger.info(
    "SearchAgent complete — %d queries, %d sources, %d evidence fragments",
    len(state.search_queries),
    len(state.sources),
    len(state.evidence),
)

In [ ]:
# ── Inspect Search Results ────────────────────────────────────────────────────
import pandas as pd

print(f"Generated queries:\n" + "\n".join(f"  {i+1}. {q}" for i, q in enumerate(state.search_queries)))

sources_df = pd.DataFrame([
    {"#": i, "title": s.title[:55], "url": s.url[:65], "images": len(s.images)}
    for i, s in enumerate(state.sources)
])
print(f"\nSources ({len(state.sources)}):")
print(sources_df.to_string(index=False))

print(f"\nEvidence sample (first 5 of {len(state.evidence)}):")
for ev in state.evidence[:5]:
    src_title = (
        state.sources[ev.source_index].title[:45]
        if ev.source_index < len(state.sources) else "?"
    )
    print(f"  [{ev.source_index}] {ev.claim[:80]}")
    print(f"       → {src_title}")

### Stage 2 — SynthesisAgent

Groups evidence fragments into coherent themes and identifies contradictions
between sources. Each theme carries a `Confidence` level (`high` / `medium` / `low`).

In [ ]:
# ── Stage 2: SynthesisAgent ───────────────────────────────────────────────────
from app.agents import synthesis

state = synthesis.run(state, on_event=_on_event)

logger.info(
    "SynthesisAgent complete — %d themes, %d contradictions",
    len(state.themes),
    len(state.contradictions),
)

In [ ]:
# ── Inspect Synthesis Results ─────────────────────────────────────────────────
print(f"Themes ({len(state.themes)}):")
for i, t in enumerate(state.themes):
    print(f"\n  [{i}] {t.name}  (confidence: {t.confidence.value})")
    print(f"       {t.summary}")
    print(f"       Evidence indices: {t.evidence_indices}")

if state.contradictions:
    print(f"\nContradictions ({len(state.contradictions)}):")
    for c in state.contradictions:
        print(f"  — {c.description}")
        print(f"    Resolution: {c.resolution}")
else:
    print("\nNo contradictions identified.")

### Stage 3 — ReportAgent

Generates a section outline with the LLM, then writes a full academic-style
Markdown report. Sources are deduplicated by URL and every claim is numbered
with inline citations `[n]`.

In [ ]:
# ── Stage 3: ReportAgent ──────────────────────────────────────────────────────
from app.agents import report as report_agent

state = report_agent.run(state, on_event=_on_event)

logger.info(
    "ReportAgent complete — outline: %d sections, report: %d chars",
    len(state.report_outline),
    len(state.final_report),
)

In [ ]:
# ── Preview Report ────────────────────────────────────────────────────────────
from IPython.display import Markdown

print("Outline:")
for s in state.report_outline:
    print(f"  • {s}")
print()

Markdown(state.final_report)

### Stage 4 — Evaluator

Uses the LLM to score the report against the evidence on four dimensions
(all in the range 0 – 1):

| Dimension | Higher is better? | What it measures |
|---|---|---|
| `coverage` | Yes | Does the report address the research question? |
| `faithfulness` | Yes | Are claims supported by the provided evidence? |
| `hallucination_rate` | **No** | Proportion of unsupported/overstated claims |
| `usefulness` | Yes | Is the report clear and decision-supportive? |

In [ ]:
# ── Stage 4: Evaluator ────────────────────────────────────────────────────────
from app.agents import evaluator

state = evaluator.run(state, on_event=_on_event)

if state.evaluation:
    ev = state.evaluation
    print(f"Coverage:          {ev.coverage:.2f}")
    print(f"Faithfulness:      {ev.faithfulness:.2f}")
    print(f"Hallucination:     {ev.hallucination_rate:.2f}  (lower is better)")
    print(f"Usefulness:        {ev.usefulness:.2f}")
    print(f"\nReasoning:\n{ev.reasoning}")

---
## Full Pipeline — `run_pipeline()`

`run_pipeline()` wires the four stages together, fires `on_stage` / `on_event`
callbacks at each transition, and optionally writes `<slug>_report.md` and
`<slug>_state.json` to `output_dir`.

In [ ]:
# ── Full Pipeline ─────────────────────────────────────────────────────────────
from app.core.pipeline import run_pipeline
from app.models.api import JobStatus

pipeline_events: list[tuple[str, dict]] = []

def _on_stage(status: JobStatus) -> None:
    print(f"  → {status.value.upper()}", flush=True)

def _on_pipeline_event(name: str, data: dict | None = None) -> None:
    pipeline_events.append((name, data or {}))

final_state = run_pipeline(
    question=RESEARCH_QUESTION,
    output_dir="output",   # writes <slug>_report.md + <slug>_state.json
    on_stage=_on_stage,
    on_event=_on_pipeline_event,
)

print()
print(final_state.summary())

In [ ]:
# ── Evaluation Score Chart ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

scores = final_state.evaluation
if scores:
    labels  = ["Coverage", "Faithfulness", "Hallucination\n(lower = better)", "Usefulness"]
    values  = [scores.coverage, scores.faithfulness, scores.hallucination_rate, scores.usefulness]
    colors  = ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6"]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=1.5)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score  (0 – 1)")
    ax.set_title("Report Quality Scores")
    ax.axhline(0.7, color="gray", linestyle="--", linewidth=0.8, label="0.7 reference")
    ax.legend()

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.2f}",
            ha="center", va="bottom", fontsize=11,
        )

    plt.tight_layout()
    plt.show()
    print(f"\nReasoning: {scores.reasoning}")
else:
    print("No evaluation scores available.")

In [ ]:
# ── Pipeline Event Log ────────────────────────────────────────────────────────
# Shows every event emitted by agents during the pipeline run.
events_df = pd.DataFrame([
    {"event": name, "keys": ", ".join(data.keys()) if data else ""}
    for name, data in pipeline_events
])
print(events_df.to_string(index=False))

---
## Export

`run_pipeline()` already wrote `<slug>_report.md` and `<slug>_state.json`
to `output/`. The cell below additionally renders an HTML version for sharing.

In [ ]:
# ── Export to HTML ────────────────────────────────────────────────────────────
from markdown import markdown
from pathlib import Path

_HTML_TEMPLATE = """<!DOCTYPE html>
<html lang=\"en\"><head><meta charset=\"utf-8\">
<style>
  body       {{ font-family: sans-serif; line-height: 1.6; max-width: 900px; margin: auto; padding: 40px; color: #333; }}
  h1         {{ color: #2c3e50; border-bottom: 2px solid #eee; }}
  h2         {{ color: #34495e; margin-top: 30px; }}
  pre        {{ background: #f4f4f4; padding: 10px; border-radius: 5px; overflow-x: auto; }}
  blockquote {{ border-left: 5px solid #eee; padding-left: 20px; font-style: italic; }}
</style></head><body>
{body}
</body></html>"""


def export_to_html(report_markdown: str, output_path: str | Path = "output/report.html") -> None:
    """Convert a Markdown report to a styled HTML file and write it to disk.

    Args:
        report_markdown: Markdown content to convert.
        output_path: Destination file path for the HTML output.
    """
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        _HTML_TEMPLATE.format(body=markdown(report_markdown)),
        encoding="utf-8",
    )
    logger.info("HTML report written to %s", path)


export_to_html(final_state.final_report, "output/report.html")

# List all generated output files
out_dir = Path("output")
print("Output files:")
for f in sorted(out_dir.iterdir()):
    print(f"  {f.name:<45} {f.stat().st_size:>10,} bytes")